# PropertyFinder RecSys  — Complete Code Walkthrough


---

## System Overview

The PropertyFinder Recommendation & Search system is built as a **5-lane pipeline architecture** with clear separation of concerns. Each lane maps to specific code files, infrastructure, and responsibilities.

```
┌──────────────────────────────────────────────────────────────────────┐
│  LANE 1: Data & ETL        →  redshift_data.py (Lambda)            │
│  LANE 2: ML Training        →  sagemaker_pipeline.py + pf_features │
│  LANE 3: Storage & Indexing →  seed_data.py + OpenSearch + Redis    │
│  LANE 4: AI Serving         →  search_api / recsys_api / ranking   │
│  LANE 5: Hydration          →  BFF Gateway (not our code)          │
│                                                                      │
│  Supporting: docker-compose.yml, Dockerfile, opensearch_mapping.json │
│  Single source of truth: pf_features.py                              │
└──────────────────────────────────────────────────────────────────────┘
```

The fundamental design principle is **Thin Payload**: our AI services return only `{ "id": "...", "rank_score": 0.95 }` (~200 bytes for 8 properties). The BFF Gateway hydrates these IDs from the primary SQL database to build the full ~30KB proto response with images, agent details, contacts, and pricing. This means our AI servers never store or serve stale business data, and our compute stays free for ML inference.

---

## LANE 1: Data & ETL — `redshift_data.py`

**What it does:** AWS Lambda function that extracts raw data from Redshift and writes Parquet files to S3.

**When it runs:** Scheduled via EventBridge — full extract daily, hourly delta for new listings.

**How it works:**

```
Redshift (Source of Truth)
    │
    ├─→ Inventory SQL (6 CTEs)
    │     ├─ unique_amenities     → deduplicated amenity codes per listing
    │     ├─ listing_popularity   → time-decayed event scores (90-day window)
    │     ├─ active_listings      → online listings, 6mo recency (24mo for off-plan)
    │     ├─ valid_prices         → numeric validation, non-null, positive
    │     ├─ geo_data             → coordinates, location names, paths
    │     └─ Final JOIN           → all fields merged into one wide table
    │
    ├─→ Interactions SQL (full runs only)
    │     ├─ 12-month user event history
    │     ├─ Weighted scoring: view=1, save=5, lead_click=10, lead_send=50
    │     ├─ Time decay: 0.98^days for high-value, 0.95^days for rentals
    │     └─ Rent capped at 90 days, sale at 365 days
    │
    └─→ UNLOAD to S3 as Parquet
          s3://pf-recsys-prod/gold/snapshot_date=YYYY-MM-DD/
```

**Key design decisions:**
- UNLOAD with `PARALLEL ON` for speed on large datasets
- Parquet format for columnar compression and type safety
- Delta mode (`is_hourly_delta=True`) adds a 2-hour time filter to inventory SQL only — interactions are only extracted on full runs since they're aggregated over months
- Polling loop with 14-minute safety timeout for long-running Redshift queries

**Output files:**
- `inventory/*.parquet` — all active listings with features
- `interactions/*.parquet` — user×listing interaction scores (full run only)

---

## LANE 2: ML Training — `sagemaker_pipeline.py` + `pf_features.py`

### `pf_features.py` — Single Source of Truth

This file is imported by EVERY other file. It defines three things that must stay in sync:

**1. MODEL_FEATURES (12 features):**
```
feature_annual_rent, feature_sale_price, is_sale, price_per_sqft,
listing_level_score, super_agent_score, popularity_score,
bedrooms, bathrooms, category_id, property_type_id, days_active
```

**2. `preprocess_for_model(df)`** — Feature engineering function used identically during training AND serving:
- Derives `is_sale` from `category_id` (1,3 = sale)
- Computes `feature_annual_rent` using price × rent_multiplier (daily→×365, weekly→×52, etc.)
- Computes `price_per_sqft` for value comparison
- Maps `listing_level` strings (premium/featured/standard) to numeric scores
- Calculates `days_active` from listing_date
- Fills all missing values with 0.0

**3. `build_rich_description(row)`** — Generates natural language text for vector embeddings:
```
"2 Bedroom Apartment in Dubai Marina, Jumeirah Beach Residences.
 Price: premium yearly. Amenities: Shared Pool, Covered Parking, Balcony."
```

This text is what the sentence-transformer model encodes into 384-dimensional vectors.

### `sagemaker_pipeline.py` — The Brain Factory

Runs on SageMaker (or locally). Three sequential stages:

**Stage 1: `generate_vectors()`**
```
Listings DataFrame
    │
    ├─→ build_rich_description() for each row
    ├─→ SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
    │     Model: 384-dim multilingual embeddings
    │     Batch size: 512
    │     Supports: English + Arabic property descriptions
    │
    └─→ property_vector column added to DataFrame
         Each vector = 384 floats representing semantic meaning
```

**Stage 2: `compute_user_dna()`**
```
For each user_id:
    │
    ├─→ Collect all property_vectors they interacted with
    ├─→ Weight by interaction_score
    ├─→ KMeans(n_clusters=2) → TWO intent vectors
    │     Why 2? Users often have dual intent:
    │       vector_1 = "Studio apartments in Marina" (primary browsing)
    │       vector_2 = "3BR villas in Arabian Ranches" (aspirational)
    │
    └─→ Output: user_vectors.parquet
         { user_id, vector_1: [384 floats], vector_2: [384 floats] }
```

If a user has fewer than 3 interactions, both vectors are set to the weighted average (graceful degradation — no clustering on tiny data).

**Stage 3: `train_ranker()`**
```
Merge interactions × listings on property_listing_id
    │
    ├─→ preprocess_for_model() — same function as serving
    ├─→ Labels: interaction_score clipped to [0, 31], ceiling'd to int
    ├─→ Query groups: factorized user_id (XGBoost learns per-user ranking)
    │
    └─→ XGBRanker(objective='rank:ndcg', ndcg_exp_gain=False)
         300 trees, depth 6, learning_rate 0.02
         Output: brain.pkl = { version, model, features }
```

**Stage 4: `export_artifacts()`**
```
Saves to artifacts/ directory:
    ├─→ inventory.parquet     (listings WITH vectors — ready for OpenSearch)
    ├─→ user_vectors.parquet  (User DNA — ready for Redis)
    └─→ brain.pkl             (XGBoost model — ready for ranking_api)
```

**Delta pipeline:** `run_delta_pipeline()` — hourly. Only re-embeds new listings, delta-upserts to OpenSearch. No retraining (model retrains daily).

---

## LANE 3: Storage & Indexing — `seed_data.py` + `opensearch_mapping.json`

### `opensearch_mapping.json` — Index Schema

```
OpenSearch Index: pf-inventory-v1
├─ Settings
│   ├─ knn: true (enables vector search)
│   ├─ knn.algo_param.ef_search: 100 (accuracy vs speed tradeoff)
│   ├─ 3 shards, 1 replica
│   └─ refresh_interval: 30s
│
└─ Mappings (key fields)
    ├─ property_listing_id: keyword (exact match, used as _id)
    ├─ category_id: integer (1=ResidentialSale, 2=ResidentialRent, 3=CommSale, 4=CommRent)
    ├─ price: float, price_period: keyword
    ├─ location_coordinates: geo_point (for radius search)
    ├─ amenities: integer (array of amenity IDs)
    ├─ listing_date: date (strict_date_optional_time — requires 'T' separator)
    ├─ quality_score, popularity_score, super_agent_score: float
    │
    └─ property_vector: knn_vector
         ├─ dimension: 384
         ├─ method: HNSW (Hierarchical Navigable Small World)
         ├─ space_type: cosinesimil
         └─ engine: lucene
```

### `seed_data.py` — Index Loader

Two modes of operation:

**Full Seed (Nightly) — Blue-Green Deployment:**
```
1. Acquire distributed Redis lock (UUID-based, 30min timeout)
2. Create NEW timestamped index: pf-inventory-v1-20260227-160202
3. prepare_document() for each row:
   ├─ Convert dates to ISO 8601 (space → 'T')
   ├─ Parse amenities string → integer array
   ├─ Build geo_point from lat/lon
   └─ Convert numpy vectors to Python lists
4. Bulk index all documents (500-doc chunks)
5. Safety check: if <90% indexed, ABORT (don't swap alias)
6. Atomic alias swap: pf-inventory-v1 → new index
7. Delete old index
8. Force refresh (count is immediately accurate)
9. Release lock
```

**Delta Upsert (Hourly):**
```
1. Acquire lock
2. Bulk upsert into EXISTING index behind alias
3. Release lock
```

**Redis DNA Loading:**
```
Read user_vectors.parquet
For each user:
    SET user_dna:{user_id} → JSON({vector_1, vector_2})
    TTL = 259,200 seconds (3 days)
    Pipeline batch = 500 commands
```

---

## LANE 4: AI Serving — Three Microservices

### Architecture Decision: Why 3 Services?

```
┌─────────────┐     ┌─────────────┐     ┌──────────────┐
│ search_api  │────→│             │     │              │
│   :8000     │     │ ranking_api │     │  OpenSearch   │
│ 4 workers   │────→│   :8002     │     │  (vectors)   │
├─────────────┤     │ 1 worker    │     ├──────────────┤
│ recsys_api  │────→│ brain.pkl   │     │   Redis      │
│   :8001     │     │ in memory   │     │  (DNA cache) │
│ 4 workers   │────→│             │     │              │
└─────────────┘     └─────────────┘     └──────────────┘
```

- **ranking_api** is isolated because `brain.pkl` is memory-intensive. 1 worker = 1 copy in RAM. Multiple workers would OOM.
- **search_api** and **recsys_api** are I/O-bound (network calls to OpenSearch/Redis/ranking). 4 async workers each handle high concurrency.
- Circuit breaker is **shared via Redis** across all workers so one worker's failures protect the whole fleet.

---

### Service 1: `ranking_api.py` — Model-as-a-Service (Port 8002)

**Endpoints:**
- `GET /health` — returns brain status, version, uptime
- `POST /rank` — accepts property features, returns ranked scores
- `POST /admin/reload-brain` — hot-reload brain.pkl without restart

**Request flow:**
```
POST /rank { "properties": [ {price: 500000, bedrooms: 2, ...}, ... ] }
    │
    ├─→ _predict_sync() runs in asyncio.to_thread (CPU-bound, won't block event loop)
    │     ├─→ pd.DataFrame(properties)
    │     ├─→ preprocess_for_model(df)  ← SAME function as training
    │     ├─→ df.reindex(columns=MODEL_FEATURES)  ← exact feature alignment
    │     └─→ brain['model'].predict(X)
    │
    └─→ Returns: { "ranked_results": [ {property_listing_id, rank_score}, ... ] }
```

**Resilience:**
- If brain.pkl fails to load → health returns 503, /rank returns empty
- If pickle is corrupt → caught at startup, logged, service stays up (degraded)
- Feature mismatch between brain.pkl and MODEL_FEATURES → brain rejected, logged

---

### Service 2: `recsys_api.py` — Recommendations (Port 8001)

**Endpoint:** `GET /internal/v1/recommendations/{id}?locale=en`

**Full request flow (6 steps matching architecture diagram):**

```
Step 1: Check Redis cache → return immediately if hit

Step 2: Fetch anchor listing from OpenSearch
         ├─→ Direct GET by _id (fast path)
         └─→ Fallback: term search on property_listing_id (handles type mismatches)

Step 3: Personalization (The Brain)
         ├─→ GET user_dna:{user_id} from Redis
         ├─→ If DNA exists:
         │     ├─ Compare anchor_vector vs vector_1 and vector_2 (cosine similarity)
         │     ├─ Pick closest persona (studio-browser vs villa-browser)
         │     └─ Blend: 65% anchor + 35% user_persona → search_vector
         └─→ If no DNA (cold start): use anchor_vector as-is

Step 4: KNN Vector Search
         POST to OpenSearch:
         {
           "query": {
             "knn": {
               "property_vector": {
                 "vector": search_vector,    ← 384 floats
                 "k": 60,                    ← fetch 60 candidates
                 "filter": {
                   "must": [{ "term": { "category_id": 1 } }],
                   "must_not": [{ "term": { "property_listing_id": anchor_id } }]
                 }
               }
             }
           }
         }

Step 5: ML Re-Ranking
         ├─→ Extract ranking-relevant fields from 60 hits
         ├─→ POST to ranking_api:8002/rank
         ├─→ Sort by rank_score descending
         └─→ Fallback: sort by quality_score if circuit breaker is OPEN

Step 6: Diversity Filter + Thin Response
         ├─→ Max 2 properties per agent (prevent agent flooding)
         ├─→ Max 3 properties per location (prevent location bias)
         ├─→ Take top 8 (default limit)
         └─→ Return: { "properties": [{ "id": "...", "rank_score": 0.95 }, ...] }
```

---

### Service 3: `search_api.py` — SERP Search (Port 8000)

**Endpoints:**
- `GET /internal/v1/search/{locale}` — query params matching proto contract
- `POST /internal/v1/search/{locale}` — JSON body for complex filters

**Supported sort modes (19 total):**
- **ML-ranked:** mlWeights, featured, caratRankingVariantA/B, superAgentVariantA/B/C, etc.
- **OpenSearch-sorted:** priceAsc, priceDesc, bedroomAsc, bedroomDesc, newest, freshnessDate

**Full request flow:**

```
Step 1: Cache check (Redis, 60s TTL, fingerprinted by filters+sort+page)

Step 2: Build OpenSearch query from filters
         ├─→ category_id, property_type_ids, bedrooms, price range
         ├─→ amenities (name → ID mapping via AMENITY_MAP)
         ├─→ geo_distance filter (lat/lon + radius)
         ├─→ completion_status, furnished_flag, is_verified
         └─→ Price period auto-derived from category (rent=yearly, sale=null)

Step 3: 4-Tier Fallback (Zero-Result Protection)
         Tier 0: Exact query          → if results > 0, done
         Tier 1: Drop amenities       → retry
         Tier 2: Expand price +20%    → retry
         Tier 3: Drop geo filter      → retry (user always sees results)

Step 4: ML Ranking (only for mlWeights/featured sorts)
         ├─→ Fetch 3× the requested limit (e.g., 150 for limit=50)
         ├─→ POST to ranking_api:8002/rank
         ├─→ Sort by ML rank_score
         └─→ Fallback: quality_score if circuit breaker OPEN

Step 5: Thin Payload Response
         {
           "meta": { "page": 1, "total_count": 1523, "per_page": 50, "page_count": 31 },
           "properties": [{ "id": "801", "rank_score": 0.9523 }, ...],
           "fallback": false,
           "fallback_tier": null
         }
```

---

## LANE 5: Hydration — BFF Gateway (Not Our Code)

This is the existing PropertyFinder backend. Our AI services are **internal-only** — they sit behind the BFF.

```
Client App
    │
    ├─→ GET /property-api/web/v1/en/more-properties/12345  (proto contract)
    │
    ↓
BFF Gateway
    ├─→ GET /internal/v1/recommendations/12345               (our API)
    │     Returns: [{ id: "991", rank_score: 0.95 }, ...]    (~200 bytes)
    │
    ├─→ SELECT * FROM properties WHERE id IN (991, 992, ...)  (SQL hydration)
    │     Returns: images, agent, broker, pricing, contacts    (~30KB)
    │
    └─→ Assemble PLPSimilarProperty proto
         rank_score = from AI payload
         All other 70+ fields = from SQL
         Order preserved from AI ranking
```

**Why this matters:** A price change, a sold listing, or a new agent photo is reflected instantly without reindexing. AI never serves stale business data.

---

## Resilience & Fault Tolerance

| Failure Scenario | What Happens | Code Location |
|---|---|---|
| **Ranking API down** | Circuit breaker opens after 5 failures. Search/RecSys auto-fallback to `quality_score` sorting. Recovers after 30s cooldown. | `RedisCircuitBreaker` class in both `search_api.py` and `recsys_api.py` |
| **Zero search results** | 4-tier filter relaxation: drop amenities → expand price → drop geo. User always sees results. | `_execute_search()` fallback loop in `search_api.py` |
| **New user (cold start)** | No DNA in Redis → skip blending → pure item-to-item KNN similarity. Still relevant from first click. | `compute_search_vector()` in `recsys_api.py` |
| **Stale price / sold listing** | AI returns only IDs. BFF hydrates from live SQL — changes reflected instantly. | Architecture-level (thin payload design) |
| **OpenSearch indexing failure** | If <90% docs indexed, alias swap is aborted. Live index stays untouched. | `seed_opensearch_full()` in `seed_data.py` |
| **Concurrent seeding** | Redis distributed lock with UUID ownership. Atomic release via Lua script. 30-min timeout. | `_acquire_lock()` / `_release_lock()` in `seed_data.py` |
| **brain.pkl corrupt** | Caught at load time. Ranking API stays up in degraded mode (503 health). Other services fallback to quality_score. | `load_model()` in `ranking_api.py` |

---

## Infrastructure — `docker-compose.yml`

| Service | Image/Build | Workers | Port | Key Config |
|---|---|---|---|---|
| ranking-api | Dockerfile.api | **1** (brain.pkl memory) | 8002 | `BRAIN_PATH=/app/artifacts/brain.pkl` |
| search-api | Dockerfile.api | 4 | 8000 | `OPENSEARCH_HOST=opensearch` |
| recsys-api | Dockerfile.api | 4 | 8001 | `OPENSEARCH_HOST=opensearch` |
| opensearch | opensearch:2.11.0 | — | 9200 | `1GB heap`, security disabled, single-node |
| redis | redis:7-alpine | — | 6379 | `512MB`, `volatile-lru` eviction, AOF persistence |

**Dependency chain:** opensearch + redis must be healthy → ranking-api must be healthy → search-api + recsys-api start.

**Production note:** docker-compose is for local/dev only. Production uses ECS task definitions or Kubernetes manifests with auto-scaling, ALB health checks, and proper resource limits.

---

## File Inventory

| File | Lane | Purpose |
|---|---|---|
| `redshift_data.py` | Data & ETL | Lambda: Redshift → S3 Parquet |
| `sagemaker_pipeline.py` | ML Training | Vectors + User DNA + XGBRanker |
| `pf_features.py` | Shared | Feature definitions, preprocessing, embeddings |
| `seed_data.py` | Storage | OpenSearch blue-green indexer + Redis loader |
| `opensearch_mapping.json` | Storage | Index schema with HNSW vector config |
| `ranking_api.py` | AI Serving | XGBoost inference microservice |
| `recsys_api.py` | AI Serving | KNN + personalization + diversity |
| `search_api.py` | AI Serving | SERP filters + fallback + 19 sort modes |
| `docker-compose.yml` | Infra | Local orchestration (5 services) |
| `Dockerfile` / `Dockerfile.api` | Infra | Python 3.11-slim container image |
| `requirements.txt` | Infra | Full ML stack (pipeline) |
| `requirements-api.txt` | Infra | Lean API deps (no sentence-transformers) |

---

## Production Data Flow Summary

```
DAILY (Full Retrain):
  Redshift → Lambda ETL → S3 Parquet
      → SageMaker Pipeline:
          1. generate_vectors()     → 384-dim embeddings
          2. compute_user_dna()     → dual-intent persona clusters
          3. train_ranker()         → brain.pkl (XGBRanker)
      → seed_data.py:
          4. Blue-green index swap  → OpenSearch
          5. DNA load               → Redis
      → ranking_api:
          6. Hot reload brain.pkl   → POST /admin/reload-brain

HOURLY (Delta):
  Redshift → Lambda ETL (delta mode) → S3 Parquet
      → SageMaker (delta mode):
          1. generate_vectors() for new listings only
      → seed_data.py:
          2. Delta upsert into existing index
      → New listings searchable in ~5 minutes

REAL-TIME (Per Request):
  User opens PDP → BFF → recsys_api → KNN + Ranking → Thin IDs → BFF → SQL Hydration → App
  User searches   → BFF → search_api → Filters + Ranking → Thin IDs → BFF → SQL Hydration → App
```
